#### Grafici

Se `data/processed/` è vuota, prima eseguire `uv run python -m src.data` dalla cartella del progetto.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, next(str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))   # cartella del progetto: rende importabile src

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter, PercentFormatter
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform

from src.data import load

daily, intraday, sentiment = load("daily"), load("intraday_1m"), load("sentiment")
full_name = {t.split()[0]: t for t in daily["ticker"].cat.categories}  # "AAPL" -> "AAPL UW Equity"

def price(ticker):
    """PX_LAST di un ticker indicizzato per data: basta il nome breve ("AAPL") o quello completo ("AAPL UW Equity")."""
    short = ticker.split()[0]
    if short not in full_name:
        raise ValueError(f"ticker non presente nel daily: {ticker}")
    return daily.loc[daily["ticker"] == full_name[short]].set_index("date")["PX_LAST"].rename(short)

# 8 colori in ordine fisso, distinguibili anche da chi è daltonico, e colori neutri per testo, griglia e assi
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
BLUE, REFERENCE = PALETTE[0], "#9a9992"
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
LW = 1.5
compact = FuncFormatter(lambda v, _: f"{v / 1e6:,.0f}M" if v >= 1e6 else f"{v / 1e3:,.0f}K" if v >= 1e3 else f"{v:,.0f}")
percent = PercentFormatter(1, decimals=0)
# scala sequenziale (un solo blu, dal chiaro allo scuro) per le heatmap
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])

def style(ax, title):
    ax.set_facecolor(SURFACE)
    ax.set_title(title, loc="left", color=INK, fontsize=11, pad=10)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(AXIS)
    ax.tick_params(which="both", colors=MUTED, labelsize=9, length=0)

def minute_axis(ax, index):
    """Converte gli orari (datetime.time) in minuti dalla mezzanotte, con un'etichetta per ogni ora."""
    ticks = range(10 * 60, 16 * 60 + 1, 60)
    ax.set_xticks(ticks, [f"{m // 60}:00" for m in ticks])
    return [t.hour * 60 + t.minute for t in index]

sentiment: 347 segnaposto -134217.7 convertiti in NaN


#### Panoramica dei dati

In [2]:
fig, axes = plt.subplot_mosaic(
    [["coverage", "returns"], ["volume", "bars"], ["sentiment", "sentiment"]],
    figsize=(13, 12), facecolor=SURFACE, layout="constrained")

# 1. l'universo sono i titoli dell'indice di oggi: chi è entrato dopo il 2006 manca all'inizio
ax = axes["coverage"]
coverage = daily.dropna(subset=["PX_LAST"]).groupby("date").size()
ax.plot(coverage.index, coverage.values, color=BLUE, lw=LW)
ax.fill_between(coverage.index, coverage.values, color=BLUE, alpha=0.1, lw=0)
ax.set_ylim(0, 520)
style(ax, "Daily: ticker con un prezzo, per giorno")

# 2. code grasse: i rendimenti estremi sono molto più frequenti che in una normale
ax = axes["returns"]
r = daily["ret"].dropna()  # calcolati da load(): solo tra giorni di borsa consecutivi
bins = np.linspace(-0.2, 0.2, 81)
ax.hist(r[r.abs() <= 0.2], bins=bins, color=BLUE, edgecolor=SURFACE, linewidth=0.8, label="rendimenti osservati")
x = np.linspace(-0.2, 0.2, 400)
normal = len(r) * (bins[1] - bins[0]) * np.exp(-0.5 * ((x - r.mean()) / r.std()) ** 2) / (r.std() * np.sqrt(2 * np.pi))
ax.plot(x, normal, color=REFERENCE, lw=LW, label="normale con stessa media e dev. std")
ax.set_yscale("log")
ax.set_ylim(0.5, None)
ax.yaxis.set_major_formatter(compact)
ax.xaxis.set_major_formatter(percent)
ax.legend(frameon=False, labelcolor=INK_2, fontsize=9, loc="upper right")
ax.text(0.01, 0.98, f"{(r.abs() > 0.2).sum()} rendimenti oltre ±20% non mostrati", transform=ax.transAxes, va="top", color=MUTED, fontsize=8)
style(ax, "Daily: rendimenti giornalieri, numero di casi (scala log)")

# 3. profilo della giornata: volume alto in apertura e in chiusura
#    il primo e l'ultimo minuto contengono le aste di apertura e chiusura: senza scala log schiaccerebbero il resto
ax = axes["volume"]
volume = intraday.groupby(["date", "time"], observed=True)["volume"].sum().groupby(level="time", observed=True).median()
ax.plot(minute_axis(ax, volume.index), volume.values, color=BLUE, lw=LW)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(compact)
style(ax, "Intraday: volume di tutti i ticker per minuto (mediana dei giorni, scala log)")

# 4. completezza: in quali minuti mancano più barre
ax = axes["bars"]
present = intraday.groupby("time", observed=True).size() / intraday.groupby(["ticker", "date"], observed=True).ngroups
ax.plot(minute_axis(ax, present.index), present.values, color=BLUE, lw=LW)
ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(percent)
style(ax, "Intraday: ticker-giorni con una barra in quel minuto")

# 5. copertura del sentiment nel tempo, solo mesi completi (dopo l'ultimo giorno di borsa il file è vuoto)
ax = axes["sentiment"]
s = sentiment[sentiment["date"] < daily["date"].max().replace(day=1)]
covered = s["NEWS_SENTIMENT_DAILY_AVG"].notna().groupby(s["date"].dt.to_period("M")).mean()
ax.plot(covered.index.to_timestamp(), covered.values, color=BLUE, lw=LW)
ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(percent)
style(ax, "Sentiment: ticker-giorni con NEWS_SENTIMENT_DAILY_AVG, per mese")

plt.show()

/var/folders/d4/0xvzbpl17dg7jnk6kl8nfnqw0000gn/T/ipykernel_28064/1651133255.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


#### Confronto prezzi

In [3]:
# ticker da confrontare: cambia la lista e riesegui la cella
# basta il nome breve ("AAPL") o quello completo ("AAPL UW Equity"); al massimo 8, uno per colore della palette
TICKERS = ["AAPL", "JPM", "XOM"]

if len(TICKERS) > len(PALETTE):
    raise ValueError(f"al massimo {len(PALETTE)} ticker, uno per colore della palette")

# base 100 nel primo giorno in cui tutti hanno un prezzo, così i ticker entrati dopo sono confrontabili
prices = pd.concat([price(t) for t in TICKERS], axis=1)
start = prices.dropna().index[0]
prices = prices.loc[start:] / prices.loc[start] * 100

fig, ax = plt.subplots(figsize=(13, 5), facecolor=SURFACE, layout="constrained")
for (name, s), color in zip(prices.items(), PALETTE):
    ax.plot(s.index, s.values, color=color, lw=LW, label=name)  # i buchi nella serie restano interruzioni della linea
    last = s.dropna()
    ax.annotate(name, (last.index[-1], last.iloc[-1]), xytext=(5, 0), textcoords="offset points", va="center", color=INK_2, fontsize=9)
ax.set_yscale("log")
ax.yaxis.set_major_locator(LogLocator(subs=(1, 2, 5)))  # etichette a 1, 2, 5, 10, 20, 50...
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.yaxis.set_minor_formatter(NullFormatter())
ax.legend(frameon=False, labelcolor=INK_2, fontsize=9, loc="upper left")
style(ax, f"Daily: prezzo aggiustato, base 100 al {start.date()} (scala log)")
plt.show()

/var/folders/d4/0xvzbpl17dg7jnk6kl8nfnqw0000gn/T/ipykernel_28064/2327799609.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


#### Esploratore di una coppia

In [4]:
# coppia da esplorare: cambia i due ticker e riesegui la cella
PAIR = ("KO", "PEP")
FORMATION, TRADING = 252, 126      # giorni di borsa: 12 mesi per stimare, 6 per tradare (SPEC, decisione 6)
ENTRY, EXIT, STOP = 2.0, 0.0, 3.5  # soglie sullo z-score dello spread (SPEC, decisione 5)

logp = np.log(pd.concat([price(t) for t in PAIR], axis=1).dropna())
a, b = logp.columns
if len(logp) <= FORMATION:
    raise ValueError(f"{a} e {b} hanno solo {len(logp)} giorni in comune: ne servono più di {FORMATION}")

# walk-forward: hedge ratio (OLS di log a su log b), media e dev. std dello spread si stimano sui 12 mesi
# precedenti e si applicano ai 6 successivi, così lo z-score di un giorno non usa informazioni future
z = pd.Series(np.nan, index=logp.index)
window_end = set()
for start in range(FORMATION, len(logp), TRADING):
    formation, trading = logp.iloc[start - FORMATION:start], logp.iloc[start:start + TRADING]
    beta = np.polyfit(formation[b], formation[a], 1)[0]
    spread = formation[a] - beta * formation[b]
    z.iloc[start:start + TRADING] = (trading[a] - beta * trading[b] - spread.mean()) / spread.std()
    window_end.add(trading.index[-1])

# trade: apre quando |z| supera ENTRY (vende lo spread se z > 0, lo compra se z < 0); chiude quando z torna a EXIT,
# quando |z| arriva a STOP o a fine finestra, perché la finestra successiva ha un altro hedge ratio
trades, position = [], 0
for day, value in z.dropna().items():
    if position == 0:
        if ENTRY < abs(value) < STOP and day not in window_end:
            position, opened = -np.sign(value), day
        continue
    if abs(value) >= STOP:
        reason = "stop"
    elif position * value >= -EXIT:
        reason = "convergenza"
    elif day in window_end:
        reason = "fine finestra"
    else:
        continue
    trades.append((opened, day, reason))
    position = 0
trades = pd.DataFrame(trades, columns=["apertura", "chiusura", "uscita"])
print(f"{a}/{b}: {len(trades)} trade, uscite {trades['uscita'].value_counts().to_dict()}")

fig, (top, bottom) = plt.subplots(2, 1, figsize=(13, 8), sharex=True, facecolor=SURFACE, layout="constrained")

prices = np.exp(logp - logp.iloc[0]) * 100
for name, color in zip(prices, PALETTE):
    top.plot(prices.index, prices[name], color=color, lw=LW, label=name)
    top.annotate(name, (prices.index[-1], prices[name].iloc[-1]), xytext=(5, 0), textcoords="offset points", va="center", color=INK_2, fontsize=9)
top.set_yscale("log")
top.yaxis.set_major_locator(LogLocator(subs=(1, 2, 5)))
top.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
top.yaxis.set_minor_formatter(NullFormatter())
top.legend(frameon=False, labelcolor=INK_2, fontsize=9, loc="upper left")
style(top, f"{a} e {b}: prezzo aggiustato, base 100 al {prices.index[0].date()} (scala log)")

bottom.plot(z.index, z.values, color=INK_2, lw=1, label="z-score dello spread")
for level in [-STOP, -ENTRY, ENTRY, STOP]:
    bottom.axhline(level, color=MUTED, lw=0.8)
bottom.axhline(EXIT, color=AXIS, lw=0.8)
for level, label in [(ENTRY, "apertura"), (STOP, "stop")]:
    bottom.annotate(f"±{level:g} {label}", (1, level), xycoords=("axes fraction", "data"), xytext=(4, 0), textcoords="offset points", va="center", color=MUTED, fontsize=8)
bottom.scatter(trades["apertura"], z[trades["apertura"]], s=36, color=PALETTE[2], edgecolor=SURFACE, linewidth=1.5, zorder=3, label="apertura")
bottom.scatter(trades["chiusura"], z[trades["chiusura"]], s=36, color=PALETTE[6], edgecolor=SURFACE, linewidth=1.5, zorder=3, label="chiusura")
bottom.legend(frameon=False, labelcolor=INK_2, fontsize=9, loc="upper left", ncols=3)
style(bottom, f"z-score dello spread {a} - β·{b}, stimato walk-forward ({FORMATION} giorni di stima, {TRADING} di trading)")
plt.show()

KO/PEP: 34 trade, uscite {'stop': 27, 'fine finestra': 4, 'convergenza': 3}


/var/folders/d4/0xvzbpl17dg7jnk6kl8nfnqw0000gn/T/ipykernel_28064/3981354971.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


#### Correlazioni tra titoli

In [5]:
# rendimenti giornalieri con una colonna per ticker (nome breve), calcolati da load() solo tra giorni consecutivi
returns = daily.pivot(index="date", columns="ticker", values="ret").dropna(how="all")  # il primo giorno non ha rendimenti
returns.columns = [t.split()[0] for t in returns.columns]

# correlazioni dell'ultimo anno tra i ticker con almeno il 90% dei giorni, riordinate per cluster gerarchico
# (distanza = 1 - correlazione): i titoli simili finiscono vicini e formano blocchi, cioè famiglie di candidati
WINDOW = 252
recent = returns.iloc[-WINDOW:].dropna(axis=1, thresh=int(0.9 * WINDOW))
corr = recent.corr()
order = leaves_list(linkage(squareform(1 - corr.to_numpy(), checks=False), method="average"))
corr = corr.iloc[order, order]

# correlazione media tra tutti i titoli negli ultimi 63 giorni (un trimestre), a ogni fine mese:
# quando sale per tutti (crisi) i titoli si muovono insieme e le divergenze tra coppie dicono meno
avg_corr = {}
for day in returns.index.to_series().groupby(returns.index.to_period("M")).last():
    window = returns.loc[:day].iloc[-63:].dropna(axis=1)
    if len(window) == 63:
        c = np.corrcoef(window.to_numpy().T)
        avg_corr[day] = np.nanmean(c[np.triu_indices(len(c), k=1)])
avg_corr = pd.Series(avg_corr)

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5.5), width_ratios=[1, 1.4], facecolor=SURFACE, layout="constrained")
image = left.imshow(corr, cmap=BLUES, vmin=0, vmax=1, interpolation="nearest")  # le poche correlazioni negative sono mostrate come 0
left.set_xticks([])
left.set_yticks([])
for side in left.spines.values():
    side.set_visible(False)
left.set_title(f"Correlazioni dell'ultimo anno, {len(corr)} ticker ordinati per cluster", loc="left", color=INK, fontsize=11, pad=10)
colorbar = fig.colorbar(image, ax=left, shrink=0.8)
colorbar.outline.set_visible(False)
colorbar.ax.tick_params(colors=MUTED, labelsize=9, length=0)

right.plot(avg_corr.index, avg_corr.values, color=BLUE, lw=LW)
right.set_ylim(0, None)
style(right, "Correlazione media tra tutti i titoli (63 giorni, a fine mese)")
plt.show()

# le coppie più correlate dell'ultimo anno, da guardare nell'esploratore qui sopra
# (alta correlazione non basta: la coppia va poi testata per cointegrazione)
top_pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().dropna().nlargest(10)
display(top_pairs.rename_axis(["ticker 1", "ticker 2"]).round(3).to_frame("correlazione"))

/var/folders/d4/0xvzbpl17dg7jnk6kl8nfnqw0000gn/T/ipykernel_28064/1128395530.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,,correlazione
ticker 1,ticker 2,
GOOG,GOOGL,0.996
FOX,FOXA,0.985
NWS,NWSA,0.962
NSC,UNP,0.943
CPT,MAA,0.924
DHI,PHM,0.920
AMAT,LRCX,0.907
MTB,RF,0.901
DOW,LYB,0.900


#### Notizie e movimenti di prezzo

In [6]:
# nei giorni con molte notizie rispetto al solito per quel titolo i movimenti sono più ampi?
# il confronto è dentro ogni ticker (notizie del giorno / mediana del ticker), così un titolo sempre in prima pagina non domina
# attenzione: non dice cosa causa cosa, anche un grande movimento produce notizie
news = daily[["ticker", "date", "ret"]].merge(sentiment[["ticker", "date", "NEWS_PUBLICATION_COUNT"]], on=["ticker", "date"]).dropna()
typical = news.groupby("ticker", observed=True)["NEWS_PUBLICATION_COUNT"].transform("median").clip(lower=1)
news["classe"] = pd.cut(news["NEWS_PUBLICATION_COUNT"] / typical, [-np.inf, 0.5, 1, 2, 5, np.inf],
                        labels=["≤ 0,5×", "0,5–1×", "1–2×", "2–5×", "> 5×"])
stats = news.groupby("classe", observed=True)["ret"].agg(
    mediana=lambda r: r.abs().median(), oltre_5=lambda r: (r.abs() > 0.05).mean())

period = f"{news['date'].min().year}–{news['date'].max().year}"
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor=SURFACE, layout="constrained")
for ax, column, title, fmt in zip(axes, stats, ["Rendimento assoluto mediano", "Giorni con un movimento oltre ±5%"], ["{:.2%}", "{:.1%}"]):
    bars = ax.bar(stats.index.astype(str), stats[column], color=BLUE, width=0.25)
    ax.bar_label(bars, labels=[fmt.format(v) for v in stats[column]], color=INK_2, fontsize=9, padding=3)
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=1))
    ax.set_xlabel("notizie del giorno rispetto alla mediana del ticker", color=INK_2, fontsize=9)
    style(ax, f"{title}, per intensità di notizie ({period})")
plt.show()

/var/folders/d4/0xvzbpl17dg7jnk6kl8nfnqw0000gn/T/ipykernel_28064/1901899780.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
